# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates loading, inspecting, and analyzing the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. We follow the Croissant metadata schema to explore dataset structure, fields, and perform basic exploratory data analysis.

### Dataset Source
The dataset is described by a Croissant schema at:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print core metadata
ds_md = dataset.metadata
print(f"Name: {ds_md.name}")
print(f"Description: {ds_md.description}\n")
print(f"Version: {ds_md.version}")
print(f"License: {ds_md.license}")
print(f"Published: {ds_md.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs in the Croissant dataset.

In [ ]:
# Explore the record sets and their fields/columns by @id
from pprint import pprint

record_sets = dataset.metadata.recordSet
if not record_sets:
    raise RuntimeError("No record sets found in the loaded metadata. Please check the Croissant schema.")

print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}")
    fields = rs.get('field', [])
    print(f"  Number of fields: {len(fields)}")
    if fields:
        print("  Example fields:")
        for f in fields[:5]:  # Show up to 5 fields
            print(f"    - {f['@id']} : {f.get('name', '')}")
    columns = rs.get('column', [])
    if columns:
        print("  Example columns (if different from field):")
        for c in columns[:5]:
            print(f"    - {c['@id']} : {c.get('name', '')}")
    print('---')

## 3. Data Extraction
Load data from each record set into separate DataFrames using their `@id`. (If there are multiple record sets, all will be loaded. Fields are referenced by their `@id`.)

In [ ]:
# Load all record sets as dataframes using their @id
dataframes = {}
all_record_set_ids = [rs['@id'] for rs in record_sets]

for record_set_id in all_record_set_ids:
    print(f"Loading records for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Columns for {record_set_id}: {dataframes[record_set_id].columns.tolist()}")
        display(dataframes[record_set_id].head())
    else:
        print(f"No records found for {record_set_id}.\n")

## 4. Exploratory Data Analysis (EDA)

We select one record set and a numeric field for demonstration. Filtering, normalization, and grouping are shown, referencing all fields by their Croissant `@id`.

*(Replace `<your_record_set_id>`, `<numeric_field_id>`, and `<group_field_id>` with actual `@id`s from your dataset overview above as you analyze your specific data.)*

In [ ]:
# Choose the main record set and fields you want to analyze
main_record_set_id = all_record_set_ids[0]  # Change index as needed
df = dataframes[main_record_set_id]

# Display column names and some example data
print(f"Columns in chosen record set (@id={main_record_set_id}):\n{df.columns.tolist()}")

# For example, suppose one of the columns is '@id': 'cr:field:protein_abundance'
# and another is '@id': 'cr:field:sample_type' (replace with actual name)
numeric_field_id = None
group_field_id = None
for cname in df.columns:
    # Try to select a plausible numeric field and group field
    if numeric_field_id is None and (('abundance' in cname) or ('coverage' in cname) or (df[cname].dtype in ['float64', 'int64'])):
        numeric_field_id = cname
    if group_field_id is None and ('sample' in cname or 'type' in cname or 'group' in cname):
        group_field_id = cname

if numeric_field_id is None:
    print("No numeric field inferred. Please select a numeric field column name from df.columns.")
else:
    print(f"Selected numeric field for EDA: {numeric_field_id}")

if numeric_field_id:
    # Remove obvious non-numeric
    df_numeric = df[pd.to_numeric(df[numeric_field_id], errors='coerce').notnull()]
    df_numeric[numeric_field_id] = df_numeric[numeric_field_id].astype(float)

    # Set threshold for filtering (mean or 75th percentile for demo)
    threshold = df_numeric[numeric_field_id].mean() if not df_numeric[numeric_field_id].empty else 0
    filtered_df = df_numeric[df_numeric[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} rows")
    display(filtered_df[[numeric_field_id]].head())

    # Z-normalize the selected field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()

    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df)
    else:
        print("No group field for grouping found or available.\n")

## 5. Visualization
Visualize field distributions and relationships using `matplotlib` or `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the numeric field
if numeric_field_id:
    sns.histplot(df_numeric[numeric_field_id], kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

# If a group field exists, boxplot
if group_field_id and group_field_id in df_numeric.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=df_numeric[group_field_id], y=df_numeric[numeric_field_id])
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

This notebook demonstrates how to load, explore, and perform basic analyses on Croissant-formatted datasets using `mlcroissant`. We reviewed the dataset's record sets and fields, programmatically filtered and normalized data, and generated basic visualizations—all referencing fields by their Croissant `@id` for reproducibility and schema-awareness. For more advanced analyses, consult the full field documentation in the Croissant schema (`fair2.json`) for canonical `@id` usage and refer to original publications for biological interpretation.